# Feature Engineering and Dataset Creation

**Input data:** `../data/bolted_fault/processed/` and `../data/arc_fault/processed/`

**Output data:** `X.npy`, `y.npy`

1. Convert amplitudes to p.u. by dividing by U_nom and I_nom
2. Convert angles to sin and cos
3. Calculate negative-sequence current I2
4. Calculate negative-sequence voltage U2
5. Calculate active resistance R and reactive resistance X
6. Calculate derivatives dI/dt, dU/dt, dZ/dt
7. Save flat `X.npy` `(N_rows x 44)` and `y.npy` `(N_rows,)`

In [ ]:
import json
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
# Data paths
PROCESSED_DIRS = [
    Path('../data/bolted_fault/processed'),   # bolted faults
    Path('../data/arc_fault/processed'),       # arc faults]

OUTPUT_DIR = Path('../data')

FAULT_TYPES = ['AG','BG','CG','ABG','BCG','CAG','ABCG','AB','BC','CA','ABC']

# Base values
U_NOM = 132790 * 100   # nominal voltage (100 — coefficient from the mathematical model)
I_NOM = 248 * 1000     # nominal line current (1000 — coefficient from the mathematical model)

# Small value for division-by-zero protection
EPS = 1e-6

## Feature Engineering Functions

In [ ]:
def to_pu(df: pd.DataFrame) -> pd.DataFrame:
    """Function for converting amplitudes to per-unit values"""
    df = df.copy()
    for sig in ['Ia','Ib','Ic','I0']:
        df[f'amp_{sig}'] = df[f'amp_{sig}'] / (I_NOM * np.sqrt(2))  # divide by the nominal current amplitude
    for sig in ['Ua','Ub','Uc','U0']:
        df[f'amp_{sig}'] = df[f'amp_{sig}'] / (U_NOM * np.sqrt(2))  # divide by the nominal voltage amplitude
    return df

In [ ]:
def angles_to_sincos(df: pd.DataFrame) -> pd.DataFrame:
    """Function for replacing an angle with sin and cos to account for cyclicity"""
    df = df.copy()
    for sig in ['Ia','Ib','Ic','I0','Ua','Ub','Uc','U0']:
        col = f'phi_{sig}'
        df[f'sin_{sig}'] = np.sin(df[col])
        df[f'cos_{sig}'] = np.cos(df[col])
        df = df.drop(columns=[col])  # remove the original angle
    return df

In [ ]:
def add_negative_sequence_current(df: pd.DataFrame) -> pd.DataFrame:
    """Function for calculating negative-sequence current I2
    Formula using symmetrical components:
      I2 = (1/3) * |Ia + a²*Ib + a*Ic|
      where a = e^(j*2π/3)
    """
    df = df.copy()
    a  = np.exp(1j * 2 * np.pi / 3) # 120-degree rotation operator
    a2 = a ** 2

    # Reconstruct complex current phasors
    # Recover the angle through arctan2(sin, cos)
    phi_a = np.arctan2(df['sin_Ia'], df['cos_Ia'])
    phi_b = np.arctan2(df['sin_Ib'], df['cos_Ib'])
    phi_c = np.arctan2(df['sin_Ic'], df['cos_Ic'])

    Ia = df['amp_Ia'].values * np.exp(1j * phi_a.values)
    Ib = df['amp_Ib'].values * np.exp(1j * phi_b.values)
    Ic = df['amp_Ic'].values * np.exp(1j * phi_c.values)

    I2 = (1/3) * np.abs(Ia + a2 * Ib + a * Ic)
    df['I2'] = I2
    return df

In [ ]:
def add_negative_sequence_voltage(df: pd.DataFrame) -> pd.DataFrame:
    """Function for calculating negative-sequence voltage U2
    Formula using symmetrical components:
        U2 = (1/3) * |Ua + a²*Ub + a*Uc|
        where a = e^(j*2π/3)
    """
    df = df.copy()
    a  = np.exp(1j * 2 * np.pi / 3) # 120-degree rotation operator
    a2 = a ** 2

    # Reconstruct complex current phasors
    # Recover the angle through arctan2(sin, cos)
    phi_a = np.arctan2(df['sin_Ua'], df['cos_Ua'])
    phi_b = np.arctan2(df['sin_Ub'], df['cos_Ub'])
    phi_c = np.arctan2(df['sin_Uc'], df['cos_Uc'])

    Ua = df['amp_Ua'].values * np.exp(1j * phi_a.values)
    Ub = df['amp_Ub'].values * np.exp(1j * phi_b.values)
    Uc = df['amp_Uc'].values * np.exp(1j * phi_c.values)

    U2 = (1/3) * np.abs(Ua + a2 * Ub + a * Uc)
    df['U2'] = U2
    return df

In [ ]:
def add_impedance(df: pd.DataFrame) -> pd.DataFrame:
    """Function for calculating active resistance R and reactive resistance X for each of the three phases.

    Z = U / I (complex phasor division)
    Z = R + jX

    Division-by-zero protection through EPS"""
    df = df.copy()
    for phase in ['a', 'b', 'c']:
        # Recover the angle
        phi_I = np.arctan2(df[f'sin_I{phase}'], df[f'cos_I{phase}'])
        phi_U = np.arctan2(df[f'sin_U{phase}'], df[f'cos_U{phase}'])
        
        # Reconstruct complex values
        I = df[f'amp_I{phase}'].values * np.exp(1j * phi_I.values)
        U = df[f'amp_U{phase}'].values * np.exp(1j * phi_U.values)

        denom = np.where(np.abs(I) < EPS, EPS, I)
        Z = U / denom
        df[f'R_{phase}'] = np.real(Z)
        df[f'X_{phase}'] = np.imag(Z)
    return df


In [ ]:
def add_derivatives(df: pd.DataFrame) -> pd.DataFrame:
    """Function for calculating derivatives dI/dt, dU/dt, dR/dt, dX/dt, dI2/dt, dU2/dt
    The first row is duplicated to preserve length"""
    df = df.copy()
    for sig in ['Ia', 'Ib', 'Ic', 'I0']:
        col = f'amp_{sig}'
        diff = np.diff(df[col].values, prepend=df[col].values[0])
        df[f'd{sig}_dt'] = diff

    for sig in ['Ua', 'Ub', 'Uc', 'U0']:
        col = f'amp_{sig}'
        diff = np.diff(df[col].values, prepend=df[col].values[0])
        df[f'd{sig}_dt'] = diff

    for phase in ['a', 'b', 'c']:
        for comp in ['R', 'X']:
            col = f'{comp}_{phase}'
            diff = np.diff(df[col].values, prepend=df[col].values[0])
            df[f'd{comp}{phase}_dt'] = diff

    # negative-sequence derivatives
    for sig in ['I2', 'U2']:
        diff = np.diff(df[sig].values, prepend=df[sig].values[0])
        df[f'd{sig}_dt'] = diff

    return df

## Dataset Assembly

In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """Applies all feature-engineering steps sequentially."""
    df = to_pu(df)
    df = angles_to_sincos(df)
    df = add_negative_sequence_current(df)
    df = add_negative_sequence_voltage(df)

    # --- DERIVATIVES DISABLED ---
    # To restore them, uncomment the lines below and set FEATURES=42 in 4_train_models.ipynb
    df = add_impedance(df)
    # df = add_derivatives(df)
    # df = df.drop(columns=['R_a', 'X_a', 'R_b', 'X_b', 'R_c', 'X_c'])

    return df

In [ ]:
def build_dataset():
    """Reads all files, builds features, and saves:
    - X.npy             (N_rows × 46 features)
    - y.npy             (N_rows,)
    - file_boundaries.npy  (N_files × 2) — start/end indices of each file
    - file_names.json   — list of file names in the same order
    - feature_names.json — feature names
    """
    all_X, all_y = [], []
    file_boundaries = []  # (start, end) for each file
    file_names = []       # file names in the same order
    feature_names = None
    total_files = 0
    current_row = 0

    for processed_dir in PROCESSED_DIRS:
        print(f"\nProcessing: {processed_dir}")
        for fault_type in FAULT_TYPES:
            folder = processed_dir / fault_type
            if not folder.exists():
                print(f"  [!] Not found: {folder}")
                continue
            csv_files = sorted(folder.glob('*.csv'))
            for fpath in csv_files:
                df = pd.read_csv(fpath, sep=";")
                df = build_features(df)

                # Store feature names from the first file.
                if feature_names is None:
                    feature_names = [c for c in df.columns if c != 'label']

                feature_cols = [c for c in df.columns if c != 'label']
                X_file = df[feature_cols].values.astype(np.float32)
                y_file = df['label'].values.astype(np.int64)

                n_rows = len(X_file)
                file_boundaries.append((current_row, current_row + n_rows))
                file_names.append(f"{processed_dir.parent.name}/{fault_type}/{fpath.name}")

                all_X.append(X_file)
                all_y.append(y_file)
                current_row += n_rows
                total_files += 1

            print(f"  [{fault_type}] files: {len(csv_files)}")

    X = np.concatenate(all_X, axis=0)
    y = np.concatenate(all_y, axis=0)

    print(f"\n{'='*50}")
    print(f"Files processed: {total_files}")
    print(f"X shape: {X.shape}")
    print(f"y shape: {y.shape}")
    print(f"Balance: normal={(y==0).sum()}  fault={(y==1).sum()}  ({(y==1).mean()*100:.1f}% faults)")

    # Save arrays and metadata.
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    np.save(OUTPUT_DIR / 'X.npy', X)
    np.save(OUTPUT_DIR / 'y.npy', y)
    np.save(OUTPUT_DIR / 'file_boundaries.npy', np.array(file_boundaries))

    with open(OUTPUT_DIR / 'file_names.json', 'w', encoding='utf-8') as f:
        json.dump(file_names, f, ensure_ascii=False, indent=2)

    feature_info = {
        "feature_names": feature_names,
        "n_features": len(feature_names),
        "groups": {
            "amplitudes_pu":         [f for f in feature_names if f.startswith('amp_')],
            "sin_angles":            [f for f in feature_names if f.startswith('sin_')],
            "cos_angles":            [f for f in feature_names if f.startswith('cos_')],
            "negative_sequence":     [f for f in feature_names if f in ['I2', 'U2']],
            "impedance":             [f for f in feature_names if f.startswith('R_') or f.startswith('X_')],
            "current_derivatives":   [f for f in feature_names if f.startswith('dI')],
            "voltage_derivatives":   [f for f in feature_names if f.startswith('dU')],
            "impedance_derivatives": [f for f in feature_names if f.startswith('dR') or f.startswith('dX')],
        }
    }
    with open(OUTPUT_DIR / 'feature_names.json', 'w', encoding='utf-8') as f:
        json.dump(feature_info, f, ensure_ascii=False, indent=2)

    print(f"\nSaved to: {OUTPUT_DIR}")

    return X, y

In [ ]:
def check_correlations(X, feature_names, threshold=0.75):
    """Builds a correlation heatmap and prints highly correlated pairs."""
    df = pd.DataFrame(X, columns=feature_names)
    corr = df.corr()

    plt.figure(figsize=(20, 16))
    sns.heatmap(corr, cmap='coolwarm', center=0, vmin=-1, vmax=1,
                xticklabels=feature_names, yticklabels=feature_names)
    plt.title('Feature Correlation')
    plt.tight_layout()
    plt.show()

    print(f'\nPairs with |correlation| > {threshold}:')
    found = False
    for i in range(len(feature_names)):
        for j in range(i + 1, len(feature_names)):
            if abs(corr.iloc[i, j]) > threshold:
                print(f'  {feature_names[i]} — {feature_names[j]}: {corr.iloc[i, j]:.3f}')
                found = True
    if not found:
        print('  No such pairs found!')

In [ ]:
X, y = build_dataset()

with open(OUTPUT_DIR / 'feature_names.json', encoding='utf-8') as f:
    feature_names = json.load(f)['feature_names']

check_correlations(X, feature_names)